# TechMind — Entrenamiento del modelo
Entrenamiento sobre `data/arxiv_cs_clean.csv`, generado por `prepare_dataset.py`.

In [ ]:
from pathlib import Path
import joblib, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / 'data' / 'arxiv_cs_clean.csv').dropna(subset=['titulo', 'texto', 'categoria'])
df['contenido'] = df['titulo'].astype(str) + ' ' + df['texto'].astype(str)
df['categoria'].value_counts()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df['contenido'], df['categoria'], test_size=.2, random_state=42, stratify=df['categoria'])
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=50000)
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)
model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(x_train_tfidf, y_train)
predictions = model.predict(x_test_tfidf)
metrics = {'accuracy': accuracy_score(y_test, predictions), 'precision_weighted': precision_score(y_test, predictions, average='weighted', zero_division=0), 'recall_weighted': recall_score(y_test, predictions, average='weighted', zero_division=0), 'f1_weighted': f1_score(y_test, predictions, average='weighted', zero_division=0)}
print(metrics)
print(classification_report(y_test, predictions, zero_division=0))
confusion_matrix(y_test, predictions)

In [ ]:
(ROOT / 'models').mkdir(exist_ok=True)
joblib.dump(model, ROOT / 'models' / 'modelo.joblib')
joblib.dump(vectorizer, ROOT / 'models' / 'vectorizador.joblib')
print('Modelos exportados')